## Day 1: WiFi + Ultrasonic Sensor Pipeline

**Goal:** Get the ultrasonic sensor's readings from the Pico 2 W to ROS2 wireslessly

**What I did:** Built a pipeline where the robot's ultrasonic sensor reading gets picked up by the Pico 2 W, sent over WiFi, and repulished as ROS2 topic:
Ultrasonic sensor → Pico 2 W (WiFi/TCP server) → ROS2 node (TCP client) → /ultrasonic_distance topic

**Why this approach:** Needed the sensor reading available on the ROS2 network in a way that will scale to more sensors later, so I built a lightweight bridge pattern (Pico sends raw readings over TCP --> a python node parses & republishes as a proper ROS2 message) rather than hardcoding anything sensor-specfic into ROS2 itself.

### Arduino sketch (Pico 2 W — WiFi + ultrasonic TCP server)
​```cpp
#include <Arduino.h>
#include <WiFi.h>
#include "Freenove_4WD_Car_For_Pico_W.h"

#define port 4002
const char *ssid_Router     = "YOUR_WIFI_NAME";
const char *password_Router = "YOUR_WIFI_PASSWORD";
WiFiServer server(port);

void setup() {
  Serial.begin(115200);
  delay(1000);
  Ultrasonic_Setup();

  WiFi.disconnect();
  WiFi.begin(ssid_Router, password_Router);
  while (WiFi.status() != WL_CONNECTED) {
    delay(500);
    Serial.print(".");
  }
  Serial.println("WiFi connected.");
  Serial.print("IP address: ");
  Serial.println(WiFi.localIP());

  server.begin(port);
}

void loop() {
  WiFiClient client = server.accept();
  if (client) {
    while (client.connected()) {
      float distance = Get_Sonar();
      client.print(String(distance) + "\n");
      delay(100);
    }
    client.stop();
  }
}
​```

### ROS2 node (Python — TCP client + publisher)
Created as a new `ament_python` package (`ultrasonic_bridge`). Connects to
the Pico's TCP server, parses each incoming distance reading, and
republishes it as a `sensor_msgs/Range` message.

​```python
import socket
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Range

class UltrasonicBridge(Node):
    def __init__(self):
        super().__init__('ultrasonic_bridge')
        self.pico_ip = '192.168.0.31'
        self.pico_port = 4002
        self.sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self.sock.connect((self.pico_ip, self.pico_port))
        self.buffer = ''
        self.publisher_ = self.create_publisher(Range, '/ultrasonic_distance', 10)
        self.timer = self.create_timer(0.05, self.read_and_publish)

    def read_and_publish(self):
        try:
            data = self.sock.recv(1024).decode('utf-8')
        except BlockingIOError:
            return
        if not data:
            return
        self.buffer += data
        while '\n' in self.buffer:
            line, self.buffer = self.buffer.split('\n', 1)
            line = line.strip()
            if not line:
                continue
            try:
                distance_cm = float(line)
            except ValueError:
                continue
            msg = Range()
            msg.header.stamp = self.get_clock().now().to_msg()
            msg.header.frame_id = 'ultrasonic_sensor'
            msg.radiation_type = Range.ULTRASOUND
            msg.field_of_view = 0.26
            msg.min_range = 0.02
            msg.max_range = 4.0
            msg.range = distance_cm / 100.0
            self.publisher_.publish(msg)

def main(args=None):
    rclpy.init(args=args)
    node = UltrasonicBridge()
    node.sock.setblocking(False)
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        pass
    finally:
        node.sock.close()
        node.destroy_node()
        rclpy.shutdown()

if __name__ == '__main__':
    main()
​```

**Result:** Fully wireless, battery-powered ultrasonic readings streaming
live into ROS2 — verified with USB fully disconnected, robot running on its
own battery pack.

**Next:** Verify topic data with `ros2 topic echo`, then add the next sensor.
